# Multimodal Task Router: DeBERTa Text + CLIP Image

Train a router that classifies an image-question pair into `chartqa`, `docvqa`, or `textvqa`.

Pipeline:

```text
question -> DeBERTa-v3 embedding
image    -> CLIP embedding
concat(text, image) -> Logistic Regression -> task_type -> backend policy
```

This replaces the question-only DeBERTa notebook and is meant to reduce DocVQA/TextVQA confusion by using visual cues.

In [ ]:
from pathlib import Path
import json
import random
import subprocess
import sys

REPO_NAME = "multi-task-moe-vlm-assistant"
REPO_URL = "https://github.com/kdnehihi/multi-task-moe-vlm-assistant.git"


def find_project_root(clone_if_missing: bool = True) -> Path:
    candidates = [
        Path.cwd(),
        *Path.cwd().parents,
        Path("/content") / REPO_NAME,
        Path("/content/drive/MyDrive") / REPO_NAME,
    ]
    for candidate in candidates:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate

    colab_target = Path("/content") / REPO_NAME
    if clone_if_missing and Path("/content").exists() and not colab_target.exists():
        print(f"Project repo not found. Cloning into {colab_target}...")
        subprocess.run(["git", "clone", REPO_URL, str(colab_target)], check=True)
        if (colab_target / "src").exists():
            return colab_target

    raise ModuleNotFoundError(
        "Could not find project root containing src/. "
        f"Clone {REPO_URL} first, or set PROJECT_ROOT manually."
    )


PROJECT_ROOT = find_project_root(clone_if_missing=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data/processed/router/multimodal_validation.jsonl"
IMAGE_DIR = PROJECT_ROOT / "data/processed/router/images"
OUTPUT_DIR = PROJECT_ROOT / "checkpoints/router/multimodal_deberta_clip_router"

TEXT_MODEL_NAME = "microsoft/deberta-v3-small"  # or "microsoft/deberta-v3-base"
IMAGE_MODEL_NAME = "openai/clip-vit-base-patch32"
SEED = 42
TEST_SIZE = 0.2
MAX_TEXT_LENGTH = 96
TEXT_BATCH_SIZE = 32
IMAGE_BATCH_SIZE = 32
MIN_CONFIDENCE = 0.65
FORCE_REBUILD_ROUTER_DATA = False
ROUTER_DATA_LIMITS = {"docvqa": 1536, "chartqa": 1536, "textvqa": 1536}

random.seed(SEED)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH exists:", DATA_PATH.exists())

## Install Dependencies In Colab If Needed

In [ ]:
# Uncomment in Colab if dependencies are missing.
# %pip install -q datasets transformers accelerate scikit-learn pandas Pillow joblib

## Prepare Multimodal Router Data

In [ ]:
# Build a small gitignored JSONL with question, task_type, and saved image_path.
# This is heavier than the question-only router because CLIP needs actual images.

from PIL import Image


def _answer_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return [str(x) for x in value]
    return [str(value)]


def _save_image(image, dataset_name: str, index: int) -> str:
    IMAGE_DIR.mkdir(parents=True, exist_ok=True)
    path = IMAGE_DIR / f"{dataset_name}_{index:05d}.jpg"
    image.convert("RGB").save(path, format="JPEG", quality=90)
    return str(path)


def _stream_router_rows(dataset_name: str, limit: int | None, split: str = "validation"):
    from datasets import load_dataset

    if dataset_name == "docvqa":
        dataset = load_dataset("lmms-lab/DocVQA", "DocVQA", split=split, streaming=True)
        for idx, row in enumerate(dataset):
            if limit is not None and idx >= limit:
                break
            yield {
                "dataset": "docvqa",
                "task_type": "docvqa",
                "split": split,
                "question_id": row.get("questionId"),
                "question": row.get("question"),
                "answers": _answer_list(row.get("answers")),
                "image_path": _save_image(row["image"], "docvqa", idx),
            }
    elif dataset_name == "chartqa":
        hf_split = "val" if split == "validation" else split
        dataset = load_dataset("HuggingFaceM4/ChartQA", split=hf_split, streaming=True)
        for idx, row in enumerate(dataset):
            if limit is not None and idx >= limit:
                break
            yield {
                "dataset": "chartqa",
                "task_type": "chartqa",
                "split": split,
                "question_id": row.get("id") or row.get("id_image"),
                "question": row.get("query") or row.get("question"),
                "answers": _answer_list(row.get("label") or row.get("answer")),
                "image_path": _save_image(row["image"], "chartqa", idx),
            }
    elif dataset_name == "textvqa":
        dataset = load_dataset("lmms-lab/textvqa", split=split, streaming=True)
        for idx, row in enumerate(dataset):
            if limit is not None and idx >= limit:
                break
            yield {
                "dataset": "textvqa",
                "task_type": "textvqa",
                "split": split,
                "question_id": row.get("question_id"),
                "question": row.get("question"),
                "answers": _answer_list(row.get("answers")),
                "image_path": _save_image(row["image"], "textvqa", idx),
            }
    else:
        raise ValueError(f"Unsupported dataset: {dataset_name}")


def build_router_jsonl(path: Path, limits: dict[str, int | None]) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    count = 0
    with path.open("w", encoding="utf-8") as handle:
        for dataset_name in ["docvqa", "chartqa", "textvqa"]:
            dataset_count = 0
            for record in _stream_router_rows(dataset_name, limits.get(dataset_name)):
                question = str(record.get("question") or "").strip()
                image_path = Path(record.get("image_path", ""))
                if not question or not image_path.exists():
                    continue
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")
                count += 1
                dataset_count += 1
            print(f"{dataset_name}: saved {dataset_count} multimodal records")
    return count


if FORCE_REBUILD_ROUTER_DATA or not DATA_PATH.exists():
    print("Router data missing or rebuild requested. Building multimodal data:", DATA_PATH)
    print("Limits:", ROUTER_DATA_LIMITS)
    total = build_router_jsonl(DATA_PATH, ROUTER_DATA_LIMITS)
    print("Saved multimodal router records:", total)
else:
    print("Using existing router data:", DATA_PATH)

## Load Records

In [ ]:
from collections import Counter
from src.data.answers import canonicalize_task_type

TASK_LABELS = ["chartqa", "docvqa", "textvqa"]
LABEL_TO_ID = {label: i for i, label in enumerate(TASK_LABELS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}


def load_router_records(path: Path) -> list[dict]:
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            question = str(row.get("question", "")).strip()
            image_path = Path(row.get("image_path", ""))
            if not question or not image_path.exists():
                continue
            task_type = canonicalize_task_type(row.get("task_type", ""))
            if task_type not in LABEL_TO_ID:
                continue
            records.append({
                "question": question,
                "image_path": str(image_path),
                "task_type": task_type,
                "label": LABEL_TO_ID[task_type],
            })
    return records

records = load_router_records(DATA_PATH)
print("records:", len(records))
print("task counts:", Counter(r["task_type"] for r in records))
records[:3]

## Split

In [ ]:
from sklearn.model_selection import train_test_split

train_records, eval_records = train_test_split(
    records,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=[r["task_type"] for r in records],
)

print("train:", len(train_records), Counter(r["task_type"] for r in train_records))
print("eval:", len(eval_records), Counter(r["task_type"] for r in eval_records))

## Load Frozen Encoders

In [ ]:
import numpy as np
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer, CLIPModel, CLIPProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("device:", DEVICE)
print("text model:", TEXT_MODEL_NAME)
print("image model:", IMAGE_MODEL_NAME)

text_tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_encoder = AutoModel.from_pretrained(TEXT_MODEL_NAME).to(DEVICE)
text_encoder.eval()

image_processor = CLIPProcessor.from_pretrained(IMAGE_MODEL_NAME)
image_encoder = CLIPModel.from_pretrained(IMAGE_MODEL_NAME).to(DEVICE)
image_encoder.eval()


def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
    summed = (last_hidden_state * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-6)
    return summed / denom

## Encode Text + Image

In [ ]:
def encode_texts(questions: list[str], batch_size: int = TEXT_BATCH_SIZE) -> np.ndarray:
    vectors = []
    with torch.no_grad():
        for start in range(0, len(questions), batch_size):
            batch_questions = questions[start:start + batch_size]
            encoded = text_tokenizer(
                batch_questions,
                padding=True,
                truncation=True,
                max_length=MAX_TEXT_LENGTH,
                return_tensors="pt",
            )
            encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
            outputs = text_encoder(**encoded)
            pooled = mean_pool(outputs.last_hidden_state, encoded["attention_mask"])
            vectors.append(pooled.detach().cpu().numpy())
    return np.concatenate(vectors, axis=0)


def encode_images(image_paths: list[str], batch_size: int = IMAGE_BATCH_SIZE) -> np.ndarray:
    vectors = []
    with torch.no_grad():
        for start in range(0, len(image_paths), batch_size):
            batch_paths = image_paths[start:start + batch_size]
            images = [Image.open(path).convert("RGB") for path in batch_paths]
            encoded = image_processor(images=images, return_tensors="pt")
            encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
            outputs = image_encoder.get_image_features(**encoded)
            if hasattr(outputs, "image_embeds"):
                image_features = outputs.image_embeds
            elif hasattr(outputs, "pooler_output"):
                image_features = outputs.pooler_output
            elif isinstance(outputs, (tuple, list)):
                image_features = outputs[0]
            else:
                image_features = outputs
            image_features = torch.nn.functional.normalize(image_features, dim=-1)
            vectors.append(image_features.detach().cpu().numpy())
    return np.concatenate(vectors, axis=0)


def encode_records(rows: list[dict]) -> np.ndarray:
    questions = [row["question"] for row in rows]
    image_paths = [row["image_path"] for row in rows]
    print("Encoding text...")
    text_vectors = encode_texts(questions)
    print("Encoding images...")
    image_vectors = encode_images(image_paths)
    features = np.concatenate([text_vectors, image_vectors], axis=1)
    print("feature shape:", features.shape)
    return features

## Train Multimodal Router

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

train_labels = [r["task_type"] for r in train_records]
eval_labels = [r["task_type"] for r in eval_records]

print("train label counts:", Counter(train_labels))
print("eval label counts:", Counter(eval_labels))
train_features = encode_records(train_records)
eval_features = encode_records(eval_records)

classifier = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                C=2.0,
                max_iter=2000,
                class_weight="balanced",
                random_state=SEED,
            ),
        ),
    ]
)
classifier.fit(train_features, train_labels)

pred_labels = classifier.predict(eval_features)
metrics = {
    "eval_accuracy": accuracy_score(eval_labels, pred_labels),
    "eval_macro_f1": f1_score(eval_labels, pred_labels, average="macro"),
}
print(metrics)
print("prediction counts:", Counter(pred_labels))

## Evaluation Report

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

print("prediction counts:", Counter(pred_labels))
print(classification_report(eval_labels, pred_labels, labels=TASK_LABELS, digits=4, zero_division=0))
cm = confusion_matrix(eval_labels, pred_labels, labels=TASK_LABELS)
pd.DataFrame(cm, index=[f"true_{x}" for x in TASK_LABELS], columns=[f"pred_{x}" for x in TASK_LABELS])

## Save Router

In [ ]:
import joblib

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
text_tokenizer.save_pretrained(str(OUTPUT_DIR / "text_tokenizer"))
text_encoder.save_pretrained(str(OUTPUT_DIR / "text_encoder"))
image_processor.save_pretrained(str(OUTPUT_DIR / "image_processor"))
image_encoder.save_pretrained(str(OUTPUT_DIR / "image_encoder"))
joblib.dump(classifier, OUTPUT_DIR / "multimodal_logreg.joblib")

metadata = {
    "model_type": "deberta_clip_multimodal_logistic_regression_router",
    "text_model": TEXT_MODEL_NAME,
    "image_model": IMAGE_MODEL_NAME,
    "output_dir": str(OUTPUT_DIR),
    "data_path": str(DATA_PATH),
    "seed": SEED,
    "test_size": TEST_SIZE,
    "max_text_length": MAX_TEXT_LENGTH,
    "labels": TASK_LABELS,
    "metrics": {k: float(v) for k, v in metrics.items() if isinstance(v, (int, float))},
    "backend_policy": {
        "chartqa": "chart_dora_r8_a16_B_lr2e-5",
        "docvqa": "base_zero_shot",
        "textvqa": "textvqa_lora_only",
    },
}
(OUTPUT_DIR / "router_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("saved:", OUTPUT_DIR)
print("classifier:", OUTPUT_DIR / "multimodal_logreg.joblib")

In [ ]:
# Copy the trained multimodal router checkpoint to Google Drive.
# Run this right after the save cell above so the Streamlit/FastAPI demo can load it later.
from pathlib import Path
import shutil

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as error:
    print(f"Google Drive mount skipped or unavailable: {error}")

DRIVE_ROUTER_DIR = Path(
    "/content/drive/MyDrive/multi-task-moe-vlm-assistant/checkpoints/router/"
    "multimodal_deberta_clip_router"
)

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(f"Router output directory does not exist: {OUTPUT_DIR}")
if not (OUTPUT_DIR / "multimodal_logreg.joblib").exists():
    raise FileNotFoundError(f"Missing trained classifier: {OUTPUT_DIR / 'multimodal_logreg.joblib'}")

DRIVE_ROUTER_DIR.parent.mkdir(parents=True, exist_ok=True)
if DRIVE_ROUTER_DIR.exists():
    shutil.rmtree(DRIVE_ROUTER_DIR)
shutil.copytree(OUTPUT_DIR, DRIVE_ROUTER_DIR)

print("Copied router checkpoint to Drive:", DRIVE_ROUTER_DIR)
for required in [
    "multimodal_logreg.joblib",
    "router_metadata.json",
    "text_tokenizer",
    "text_encoder",
    "image_processor",
    "image_encoder",
]:
    path = DRIVE_ROUTER_DIR / required
    print(f"{required}: {path.exists()}")


## Predict Backend

In [ ]:
from src.routing.task_router import get_backend_for_task, base_fallback_decision


def predict_task_multimodal(question: str, image_path: str, min_confidence: float = MIN_CONFIDENCE) -> dict:
    features = np.concatenate([encode_texts([question]), encode_images([image_path])], axis=1)
    probabilities = classifier.predict_proba(features)[0]
    task_type = str(classifier.classes_[int(probabilities.argmax())])
    confidence = float(probabilities.max())
    decision = base_fallback_decision("unknown", confidence) if confidence < min_confidence else get_backend_for_task(task_type, confidence)
    return {
        "task_type": task_type,
        "confidence": confidence,
        "backend_name": decision.backend_name,
        "use_adapter": decision.use_adapter,
        "adapter_name": decision.adapter_name,
        "checkpoint_dir": decision.checkpoint_dir,
    }

sample = eval_records[0]
predict_task_multimodal(sample["question"], sample["image_path"])